# Telecom Customer Churn & Retention Analysis

This notebook reproduces the core Python/Pandas analysis. It separates data validation, descriptive findings, retention prioritization, and business interpretation.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
sys.path.insert(0, str(PROJECT_ROOT))

from scripts.analyze import segment_summary

DATA_PATH = PROJECT_ROOT / 'data' / 'processed' / 'telecom_customers_clean.csv'
df = pd.read_csv(DATA_PATH)
df.shape

## 1. Data quality
The customer identifier should be complete and unique. Negative monthly charges are preserved in the raw field and excluded from validated charge analysis.

In [ ]:
pd.Series({
    'rows': len(df),
    'unique_customer_ids': df['customer_id'].nunique(),
    'duplicate_customer_ids': df['customer_id'].duplicated().sum(),
    'invalid_monthly_charge_records': df['invalid_monthly_charge_flag'].sum(),
    'missing_population_records': df['population'].isna().sum(),
})

## 2. KPI overview

In [ ]:
churned = df[df['churn_flag'].eq(1)]
pd.Series({
    'total_customers': len(df),
    'churned_customers': len(churned),
    'churn_rate_pct': round(df['churn_flag'].mean() * 100, 2),
    'monthly_revenue_exposure_raw': round(churned['monthly_charge_raw'].sum(), 2),
    'historical_revenue_churned': round(churned['total_revenue'].sum(), 2),
    'avg_churned_charge_valid': round(churned['monthly_charge_valid'].mean(), 2),
    'avg_churned_tenure_months': round(churned['tenure_in_months'].mean(), 2),
})

## 3. Segment analysis
Segment differences are descriptive associations, not causal estimates.

In [ ]:
segment_summary(df, 'contract')

In [ ]:
contract_internet = (
    df.pivot_table(index='contract', columns='internet_type_clean', values='churn_flag', aggfunc='mean')
    .mul(100)
    .round(2)
)
contract_internet

## 4. Transparent retention priority
The rule-based score combines month-to-month contract, tenure below 12 months, validated monthly charge of at least $70, and fiber-optic service. It is an operational triage score, not a predicted probability.

In [ ]:
priority_summary = (
    df.groupby('risk_priority', observed=True)
      .agg(customers=('customer_id', 'size'), churned=('churn_flag', 'sum'), observed_churn_rate=('churn_flag', 'mean'))
      .reindex(['Low', 'Medium', 'High'])
)
priority_summary['observed_churn_rate_pct'] = (priority_summary['observed_churn_rate'] * 100).round(2)
priority_summary

## 5. Recommended next step
Use the high-priority active-customer list to define targeted contract, onboarding, and service-quality experiments. Evaluate incremental retention and contribution margin against a holdout group rather than assuming the observed segment differences are causal.